In [61]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=data['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=data['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=data['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
res='1km'
job_array=False;index_adjust=0
ocean_fraction=0.25


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [62]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(PlottingFunctions, inspect.isfunction)]
# functions

In [213]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+f'lagrangian_binary_array_{res}_{Np_str}.h5', 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    # W = f['W'][:]
    # QCQI = f['QCQI'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [363]:
# #job array setup #UNCOMMENT IF USING JOB_ARRAY
# job_array=False
# if job_array==True:
#     num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
#     job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
#     if job_id==0: job_id=1
        
#     num_parcels=len(data['time']) #total num of variables
#     job_range = num_parcels//num_jobs #number of parcels per job 
#     start_job = (job_id - 1) * job_range
#     end_job = start_job + job_range
#     if job_id==num_jobs: end_job=num_parcels-1
    
    
#     #FOR ENTRAINMENT THE PREVIOUS TIMESTEP IS ALWAYS NEEDED, SO SLICED WILL CAUSE WRAP AROUND DATA
#     #INSTEAD NP.WHERE RESULTS MUST BE SUBSETTED
#     # data=data.isel(time=slice(start_job,end_job))
#     # parcel=parcel.isel(time=slice(start_job,end_job))
    
#     # #SLICING LAGRANGIAN BINARY ARRAY
#     # A_g=A_g[slice(start_job,end_job)]
#     # A_c=A_c[slice(start_job,end_job)]
#     # Z=Z[slice(start_job,end_job)]
#     # Y=Y[slice(start_job,end_job)]
#     # X=X[slice(start_job,end_job)]

In [17]:
###########################################################################################################################################################################

In [342]:
def extend_idxs(f,case):
    out=np.sort(np.add.outer(f, np.arange(case)).ravel())

    # #OLD METHOD (SLOW)
    # if np.any(f)==True:
    #     out=np.sort(np.concatenate([np.arange(idx, idx + case-1+1) for idx in f]))
    # else: 
    #     out=f
    return out

def find_sandwiched_patterns(changes, case):
    arr=changes
    
    window_size = case + 1  # e.g., for case=2, window_size = 3
    # The interior zeros count is (window_size - 2) which is case - 1
    pattern1 = np.array([-1] + [0]*(case - 1) + [1])
    pattern2 = np.array([1] + [0]*(case - 1) + [-1])
    
    # Manually construct sliding windows
    windows = np.array([arr[i:i + window_size] for i in range(len(arr) - window_size + 1)])
    # print("Sliding windows:\n", windows) #TESTING
    
    #THE ALGORITHM
    turb_d=[]
    turb_e=[]
    count=0;max_iter=len(data['time']);
    while np.any(((windows == pattern1) | (windows == pattern2)).all(axis=1)):
        count+=1; 
        if count>=max_iter: 
            print(count)
            break
        
        next_ind = np.where(((windows == pattern1) | (windows == pattern2)).all(axis=1))[0][0]
        
        if (windows[next_ind] == pattern1).all():
            turb_d.append(next_ind)
        elif (windows[next_ind] == pattern2).all(): 
            turb_e.append(next_ind) #append to list
    
        windows[0:next_ind+(case)+1,:] = 0 #removes from windows
    
    turb_d=np.array(turb_d,dtype=int); turb_e=np.array(turb_e,dtype=int)

    #EXTEND REST OF INDEXES TO PROCESS
    turb_d=extend_idxs(turb_d,case=case)
    turb_e=extend_idxs(turb_e,case=case)
    return turb_d,turb_e

In [343]:
# # TESTING
# # changes = np.array([0,0,0,-1,1,0,0,-1,0,0,0,1,-1,0,0])
# [a,b] = find_sandwiched_patterns(changes, case=1) #<=1 in a row timesteps are removed
# print("Case matches at indices:", a,b)

# changes = np.array([0,0,0,-1,0,1,0,0,-1,0,0,1,0,-1,0,0])
# [a,b] = find_sandwiched_patterns(changes, case=2) #<=2 in a row timesteps are removed
# print("Case matches at indices:", a,b)

# changes = np.array([0,0,0,-1,0,0,1,0,0,0,0,1,0,0,-1,0,0])
# [a,b] = find_sandwiched_patterns(changes, case=3) #<=3 in a row timesteps are removed
# print("Case matches at indices:", a,b)

In [346]:
def PreProcessing(p,updraft_type):

    if updraft_type=='general':
        A=A_g
    elif updraft_type=='cloudy':
        A=A_c
    B = A[:,p]

    # Find the changes in the array
    changes = np.diff(np.concatenate(([B[0]], B)))  # Add 0s to detect edges
    # print(f'B = {B}'); print(f'changes = {changes}') #TESTING

    #Determining the Case Number
    ###### (amount of time outside of cloud to count as entrainment)
    mins_thresh=5 #5 mins
    ######
    t_per_mins=1/((data['time'][1]-data['time'][0])/1e9/60).item() #timesteps per minute (<=1)
    case=int(t_per_mins*mins_thresh)

    #Calling Algorithm and Correcting Parcel Data
    [turb_d,turb_e]=find_sandwiched_patterns(changes, case=case)
    # [turb_d,turb_e]=find_sandwiched_patterns(changes, case=3) #TESTING
    B[turb_d]=1
    B[turb_e]=0
    return B

In [ ]:
#TESTING
# # Case 1
# B=np.array([1,0,1,1,0,0,1,0])
# B=np.array([1,0,1,1,0,1,0]) 
# B=np.array([1,0,1,1,0,1,0,1])

# B=np.array([1,0,1,0,1,1,1])
# B=np.array([1,0,1,0,1,0,0])
# B=np.array([0,1,0,1,0,0,0]) 
# B=np.array([0,1,0,1,0,1,1])

# Case 2
# B=np.array([1,1,1,0,0,1,1,1,0,0,0,1,1,0,0,0])
# B=np.array([1,0,0,1,0,0,1,0,0,0])

# # Case 3
# B=np.array([1,1,1,1,0,0,0,1,1,1,1])
# p=999857; out=PreProcessing(p,updraft_type='cloudy') #TESTING
# print(f'output =  {out}\n')

# #TESTING
# count_per_row = (A_c >= 1).sum(axis=0)
# where=np.where(count_per_row > 10)[0] # Find row indices where count is greater than 10
# print(where)
# p=999697; out=PreProcessing(p,updraft_type='cloudy') #TESTING
# print(f'output =  {out}\n')

In [ ]:
# %%timeit
#RUNNING
A_g_Processed=A_g.copy() 
A_c_Processed=A_c.copy()
print('processing parcels for general updrafts')
for p in np.arange(len(parcel['xh'])):
    # if p==1000:break #TESTING
    if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
    out=PreProcessing(p,updraft_type='general'); A_g_Processed[:,p]=out
print('processing parcels for cloudy updrafts')
for p in np.arange(len(parcel['xh'])):
    # if p==1000:break #TESTING
    if np.mod(p,25000)==0: print(f"{p}/{len(parcel['xh'])}")
    out=PreProcessing(p,updraft_type='cloudy'); A_c_Processed[:,p]=out
    

#SAVING
dir3=dir+f'Project_Algorithms/Entrainment/processed_binary_arrays_'+str(mins_thresh)+f'mins_{res}_{Np_str}.h5'
with h5py.File(dir3, 'w') as h5file:
    h5file.create_dataset('A_g_Processed', data=A_g_Processed)
    h5file.create_dataset('A_c_Processed', data=A_c_Processed)
print('done')
#OLD ALGORITHM about 50 minutes for 1 million parcels (general+cloudy)
#NEW ALGORITHM ESTIMATION ==> 3.53 s ± 8.47 ms per run (tested 7 runs) ==> average = (1e6*3.53/1000)/60 = 58 minutes